In [ ]:
import numpy as np
import random

# ==========================================
# 1. 환경 정의 (6x6)
# ==========================================
class MouseCheeseWorld:
    def __init__(self):
        self.grid = [
            ['S', 'F', 'F', 'H', 'F', 'F'],
            ['F', 'H', 'F', 'F', 'F', 'H'],
            ['F', 'F', 'F', 'H', 'F', 'F'],
            ['F', 'H', 'H', 'F', 'F', 'F'],
            ['F', 'F', 'F', 'F', 'H', 'F'],
            ['H', 'F', 'F', 'F', 'F', 'G']
        ]
        self.rows = 6
        self.cols = 6
        self.start = (0, 0)
        self.goal = (5, 5)
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state_to_id(self.state[0], self.state[1])

    def state_to_id(self, r, c):
        return r * self.cols + c

    def id_to_state(self, state_id):
        return (state_id // self.cols, state_id % self.cols)

    def step(self, state_id, action):
        r, c = self.id_to_state(state_id)

        # 행동: 0:상, 1:하, 2:좌, 3:우
        if action == 0: r = max(0, r - 1)
        elif action == 1: r = min(self.rows - 1, r + 1)
        elif action == 2: c = max(0, c - 1)
        elif action == 3: c = min(self.cols - 1, c + 1)

        next_id = self.state_to_id(r, c)
        cell = self.grid[r][c]

        if cell == 'G':     return next_id, 10.0, True
        elif cell == 'H':   return next_id, -5.0, True
        else:               return next_id, -0.1, False

# ==========================================
# 2. Q-Learning
# ==========================================
def train_mouse():
    # 초기화
    env = MouseCheeseWorld()
    q_table = np.zeros((36, 4))
    episodes = 5000
    alpha = 0.01
    gamma = 0.99
    epsilon = 1.0

    # 감시 좌표
    watch_state = env.state_to_id(5, 4)

    print("🐭 6x6 미로 예시")
    print(f"{'Episode':^10} | {'Epsilon':^10} | {'Q-Value (Goal 옆칸)':^20}")
    print("-" * 48)

    for episode in range(episodes):
        state = env.reset()
        done = False

        while not done:
            if random.uniform(0, 1) < epsilon:
                action = random.randint(0, 3)
            else:
                action = np.argmax(q_table[state])

            next_state, reward, done = env.step(state, action)

            current_q = q_table[state, action]
            max_next_q = np.max(q_table[next_state])

            new_q = current_q + alpha * (reward + gamma * max_next_q - current_q)
            q_table[state, action] = new_q

            state = next_state

        # [로그 출력] 1000번마다
        if (episode + 1) % 1000 == 0:
            q_val = q_table[watch_state, 3] # 오른쪽(3) 가치
            print(f"{episode+1:^10} | {epsilon:^10.2f} | {q_val:^20.2f}")

        # 천천히 감소
        epsilon = max(0.01, epsilon * 0.9995)

    print("-" * 48)
    return env, q_table

# 시각화
def visualize_brain(env, q_table):
    arrows = ['↑', '↓', '←', '→']
    print("\n[최적 경로 지도]")
    for r in range(env.rows):
        line = []
        for c in range(env.cols):
            cell = env.grid[r][c]
            state_id = env.state_to_id(r, c)
            best_action = np.argmax(q_table[state_id])
            if cell == 'G': line.append(" 🧀")
            elif cell == 'H': line.append(" 💀")
            elif cell == 'S': line.append(" 🐭")
            else: line.append(f" {arrows[best_action]} ")
        print(" | ".join(line))

    print("\n[가치 지도 (Q-Value)]")
    for r in range(env.rows):
        line = []
        for c in range(env.cols):
            state_id = env.state_to_id(r, c)
            max_q = np.max(q_table[state_id])
            cell = env.grid[r][c]
            if cell == 'G': line.append("[GOAL]")
            elif cell == 'H': line.append("[FAIL]")
            else: line.append(f"{max_q:5.1f}")
        print(" ".join(line))

# 실행
env, q_table = train_mouse()
visualize_brain(env, q_table)

🐭 6x6 미로 예시
 Episode   |  Epsilon   |  Q-Value (Goal 옆칸)  
------------------------------------------------
   1000    |    0.61    |         0.59        
   2000    |    0.37    |         5.48        
   3000    |    0.22    |         9.88        
   4000    |    0.14    |        10.00        
   5000    |    0.08    |        10.00        
------------------------------------------------

[최적 경로 지도]
 🐭 |  ←  |  ↓  |  💀 |  ↓  |  → 
 ↓  |  💀 |  →  |  →  |  ↓  |  💀
 ↓  |  ←  |  ←  |  💀 |  →  |  ↓ 
 ↓  |  💀 |  💀 |  ↓  |  →  |  ↓ 
 →  |  ↓  |  ↓  |  ↓  |  💀 |  ↓ 
 💀 |  →  |  →  |  →  |  →  |  🧀

[가치 지도 (Q-Value)]
  8.3   2.7  -0.3 [FAIL]  -0.1  -0.1
  8.5 [FAIL]  -0.3  -0.1   0.1 [FAIL]
  8.6   3.7  -0.2 [FAIL]   0.8   2.1
  8.8 [FAIL] [FAIL]   0.2   0.3   4.6
  9.0   9.2   7.2   5.9 [FAIL]   7.8
[FAIL]   9.4   9.6   9.8  10.0 [GOAL]
